# Pipeline de Inferencia - Clasificación Jerárquica de Miopía

Este notebook implementa las **3 estrategias** de clasificación jerárquica usando los modelos ya entrenados:

1. **Top-Down (Descendente)**: Predice DCombo directamente y mapea a jerarquía
2. **Bottom-Up (Cascada)**: 3 niveles → M → MM → M1/M2
3. **Hybrid (Híbrido)**: Combina Top-Down + Bottom-Up con validación

## Requisitos
- Modelos previamente entrenados en `models/`
- Archivo de datos: `X_test.csv` o similar

## Uso
1. Ejecutar todas las celdas en secuencia
2. Los modelos se cargan automáticamente desde `models/`
3. Las predicciones se generan para las 3 estrategias
4. Se comparan los resultados

## 1. Imports y Configuración

In [1]:
# Imports
import pandas as pd
import numpy as np
import joblib
import warnings
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# Configuración
MODELS_DIR = Path('models')
CONFIDENCE_THRESHOLD = 0.70  # Umbral para estrategia Hybrid

print("✓ Imports completados")
print(f"✓ Directorio de modelos: {MODELS_DIR}")

✓ Imports completados
✓ Directorio de modelos: models


## 2. Funciones Auxiliares

In [2]:
def dcombo_to_hierarchical(dcombo_values):
    """
    Mapea valores de DCombo a las columnas M, MM y Combo.
    
    Reglas:
    - C → M=NO, MM=NO, Combo=C
    - M1 → M=SI, MM=NO, Combo=M
    - M2 → M=SI, MM=NO, Combo=M
    - MM → M=SI, MM=SI, Combo=MM
    
    Parameters:
    -----------
    dcombo_values : array-like
        Valores de DCombo
    
    Returns:
    --------
    tuple: (M, MM, Combo)
    """
    dcombo_array = np.array(dcombo_values)
    M = np.where(dcombo_array == 'C', 'NO', 'SI')
    MM = np.where(dcombo_array == 'MM', 'SI', 'NO')
    Combo = np.where(dcombo_array == 'C', 'C',
                    np.where(dcombo_array == 'MM', 'MM', 'M'))
    return M, MM, Combo


def evaluate_predictions(y_true_dict, y_pred_dict, strategy_name=""):
    """
    Evalúa predicciones para todas las columnas jerárquicas.
    
    Parameters:
    -----------
    y_true_dict : dict
        Valores reales: {'M': array, 'MM': array, 'Combo': array, 'DCombo': array}
    y_pred_dict : dict
        Predicciones: {'M': array, 'MM': array, 'Combo': array, 'DCombo': array}
    strategy_name : str
        Nombre de la estrategia para el reporte
    
    Returns:
    --------
    results : dict
        Métricas para cada columna
    """
    print(f"\n{'='*80}")
    print(f"EVALUACIÓN: {strategy_name}")
    print(f"{'='*80}")
    
    results = {}
    
    for col in ['DCombo', 'Combo', 'M', 'MM']:
        y_true = y_true_dict[col]
        y_pred = y_pred_dict[col]
        
        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
        results[col] = {
            'accuracy': acc,
            'f1_macro': f1_macro
        }
        
        print(f"\n{col}:")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  F1-Macro:  {f1_macro:.4f}")
    
    return results


print("✓ Funciones auxiliares definidas")

✓ Funciones auxiliares definidas


## 3. Cargar Modelos Entrenados

In [3]:
print("="*80)
print("CARGANDO MODELOS ENTRENADOS")
print("="*80)

# Cargar modelo Top-Down
print("\n📦 Cargando modelo Top-Down...")
model_topdown = joblib.load(MODELS_DIR / 'topdown_dcombo.pkl')
topdown_info = joblib.load(MODELS_DIR / 'topdown_info.pkl')
print(f"  ✓ Modelo: {topdown_info['model_name']}")
print(f"  ✓ F1-Macro entrenamiento: {topdown_info['best_f1_macro']:.4f}")

# Cargar label encoder si existe
label_encoder_topdown = None
if topdown_info['uses_label_encoding']:
    label_encoder_topdown = joblib.load(MODELS_DIR / 'topdown_label_encoder.pkl')
    print(f"  ✓ Label encoder cargado")

# Cargar modelos Bottom-Up (3 niveles)
print("\n📦 Cargando modelos Bottom-Up (3 niveles)...")
model_bottomup_m = joblib.load(MODELS_DIR / 'bottomup_level1_M.pkl')
model_bottomup_mm = joblib.load(MODELS_DIR / 'bottomup_level2_MM.pkl')
model_bottomup_m1m2 = joblib.load(MODELS_DIR / 'bottomup_level3_M1M2.pkl')
print(f"  ✓ Nivel 1 (M): bottomup_level1_M.pkl")
print(f"  ✓ Nivel 2 (MM): bottomup_level2_MM.pkl")
print(f"  ✓ Nivel 3 (M1/M2): bottomup_level3_M1M2.pkl")

# Cargar scores de validación Bottom-Up
scores_bottomup = joblib.load(MODELS_DIR / 'bottomup_validation_scores.pkl')
print(f"\n  Scores de validación:")
print(f"    - Nivel 1 (M): F1-Macro = {scores_bottomup['level1_M_f1_macro']:.4f}")
print(f"    - Nivel 2 (MM): F1-Macro = {scores_bottomup['level2_MM_f1_macro']:.4f}")
print(f"    - Nivel 3 (M1/M2): F1-Macro = {scores_bottomup['level3_M1M2_f1_macro']:.4f}")

# Cargar configuración Hybrid
print("\n📦 Cargando configuración Hybrid...")
hybrid_config = joblib.load(MODELS_DIR / 'hybrid_config.pkl')
print(f"  ✓ Umbral de confianza: {hybrid_config['confidence_threshold']}")
print(f"  ✓ En entrenamiento usó Bottom-Up en {hybrid_config['use_bottomup_percentage']:.2f}% de casos")

print("\n" + "="*80)
print("✓ TODOS LOS MODELOS CARGADOS EXITOSAMENTE")
print("="*80)

CARGANDO MODELOS ENTRENADOS

📦 Cargando modelo Top-Down...
  ✓ Modelo: random_forest
  ✓ F1-Macro entrenamiento: 0.7529

📦 Cargando modelos Bottom-Up (3 niveles)...
  ✓ Nivel 1 (M): bottomup_level1_M.pkl
  ✓ Nivel 2 (MM): bottomup_level2_MM.pkl
  ✓ Nivel 3 (M1/M2): bottomup_level3_M1M2.pkl

  Scores de validación:
    - Nivel 1 (M): F1-Macro = 0.7618
    - Nivel 2 (MM): F1-Macro = 0.4699
    - Nivel 3 (M1/M2): F1-Macro = 0.7601

📦 Cargando configuración Hybrid...
  ✓ Umbral de confianza: 0.7
  ✓ En entrenamiento usó Bottom-Up en 9.09% de casos

✓ TODOS LOS MODELOS CARGADOS EXITOSAMENTE
  ✓ Modelo: random_forest
  ✓ F1-Macro entrenamiento: 0.7529

📦 Cargando modelos Bottom-Up (3 niveles)...
  ✓ Nivel 1 (M): bottomup_level1_M.pkl
  ✓ Nivel 2 (MM): bottomup_level2_MM.pkl
  ✓ Nivel 3 (M1/M2): bottomup_level3_M1M2.pkl

  Scores de validación:
    - Nivel 1 (M): F1-Macro = 0.7618
    - Nivel 2 (MM): F1-Macro = 0.4699
    - Nivel 3 (M1/M2): F1-Macro = 0.7601

📦 Cargando configuración Hybrid..

## 4. Cargar Datos de Prueba

In [4]:
print("\n" + "="*80)
print("CARGANDO DATOS DE PRUEBA")
print("="*80)

# Cargar X_test (puedes cambiar por tu archivo)
# NOTA: Ajusta el nombre del archivo según tus datos
X_test = pd.read_csv('X_test.csv')
print(f"✓ X_test cargado: {X_test.shape[0]} muestras, {X_test.shape[1]} features")

# Si tienes Y_test, cárgalo para evaluar
# Si no tienes etiquetas, comenta esta sección
try:
    Y_test_df = pd.read_csv('Y_test.csv')
    y_test_dict = {
        'M': Y_test_df['M'].values,
        'MM': Y_test_df['MM'].values,
        'Combo': Y_test_df['Combo'].values,
        'DCombo': Y_test_df['DCombo'].values
    }
    print(f"✓ Y_test cargado: {len(Y_test_df)} muestras")
    print(f"\nDistribución de clases DCombo:")
    print(pd.Series(y_test_dict['DCombo']).value_counts().sort_index())
    EVALUATE = True
except FileNotFoundError:
    print("⚠ Y_test.csv no encontrado - solo se generarán predicciones (sin evaluación)")
    EVALUATE = False

print("="*80)


CARGANDO DATOS DE PRUEBA
✓ X_test cargado: 24 muestras, 49 features
⚠ Y_test.csv no encontrado - solo se generarán predicciones (sin evaluación)


## 5. Estrategia 1: TOP-DOWN (Descendente)

In [5]:
print("\n" + "="*80)
print("ESTRATEGIA 1: TOP-DOWN (Descendente)")
print("="*80)
print("\nDescripción:")
print("  - Predice DCombo directamente (4 clases: C, M1, M2, MM)")
print("  - Mapea jerárquicamente DCombo → Combo, M, MM")
print("  - Modelo: Un solo RandomForest/XGBoost")

# Predecir DCombo
print("\nGenerando predicciones...")
y_pred_dcombo_topdown = model_topdown.predict(X_test)

# Decodificar si se usó label encoding
if label_encoder_topdown is not None:
    y_pred_dcombo_topdown = label_encoder_topdown.inverse_transform(y_pred_dcombo_topdown)

# Mapear a jerarquía
M_pred_topdown, MM_pred_topdown, Combo_pred_topdown = dcombo_to_hierarchical(y_pred_dcombo_topdown)

# Diccionario de predicciones
y_pred_topdown = {
    'DCombo': y_pred_dcombo_topdown,
    'Combo': Combo_pred_topdown,
    'M': M_pred_topdown,
    'MM': MM_pred_topdown
}

print(f"✓ Predicciones Top-Down generadas: {len(y_pred_dcombo_topdown)} muestras")
print(f"\nDistribución predicha DCombo:")
print(pd.Series(y_pred_dcombo_topdown).value_counts().sort_index())

# Evaluar si hay etiquetas
if EVALUATE:
    results_topdown = evaluate_predictions(y_test_dict, y_pred_topdown, "TOP-DOWN")


ESTRATEGIA 1: TOP-DOWN (Descendente)

Descripción:
  - Predice DCombo directamente (4 clases: C, M1, M2, MM)
  - Mapea jerárquicamente DCombo → Combo, M, MM
  - Modelo: Un solo RandomForest/XGBoost

Generando predicciones...
✓ Predicciones Top-Down generadas: 24 muestras

Distribución predicha DCombo:
C     7
M1    7
M2    5
MM    5
Name: count, dtype: int64


## 6. Estrategia 2: BOTTOM-UP (Cascada)

In [11]:
print("\n" + "="*80)
print("ESTRATEGIA 2: BOTTOM-UP (Ascendente/Cascada)")
print("="*80)
print("\nDescripción:")
print("  - Nivel 1: Predice M (NO/SI)")
print("  - Nivel 2: Si M=SI → Predice MM (NO/SI)")
print("  - Nivel 3: Si M=SI y MM=NO → Predice M1 vs M2")
print("  - Modelos: 3 RandomForest optimizados independientemente")

n_samples = len(X_test)

# NIVEL 1: Predecir M
print("\nNivel 1: Prediciendo M (NO/SI)...")
m_pred = model_bottomup_m.predict(X_test)
print(f"  ✓ {(m_pred == 'SI').sum()} predicciones SI, {(m_pred == 'NO').sum()} predicciones NO")

# Inicializar arrays
mm_pred = np.array(['NO'] * n_samples, dtype=object)
dcombo_pred = np.array(['C'] * n_samples, dtype=object)

# NIVEL 2: Predecir MM (solo donde M=SI)
mask_m_si = (m_pred == 'SI')
if mask_m_si.any():
    print(f"\nNivel 2: Prediciendo MM para {mask_m_si.sum()} casos donde M=SI...")
    X_m_si = X_test.loc[mask_m_si]
    mm_pred_subset = model_bottomup_mm.predict(X_m_si)
    mm_pred[mask_m_si] = mm_pred_subset
    print(f"  ✓ {(mm_pred == 'SI').sum()} predicciones SI (MM)")
    
    # Casos donde M=SI y MM=SI → DCombo = MM
    mask_mm_si = mask_m_si & (mm_pred == 'SI')
    dcombo_pred[mask_mm_si] = 'MM'
    
    # NIVEL 3: Predecir M1 vs M2 (donde M=SI y MM=NO)
    mask_m1m2 = mask_m_si & (mm_pred == 'NO')
    if mask_m1m2.any():
        print(f"\nNivel 3: Prediciendo M1 vs M2 para {mask_m1m2.sum()} casos donde M=SI y MM=NO...")
        X_m1m2 = X_test.loc[mask_m1m2]
        m1m2_pred_subset = model_bottomup_m1m2.predict(X_m1m2)
        dcombo_pred[mask_m1m2] = m1m2_pred_subset
        
        m1_count = (dcombo_pred == 'M1').sum()
        m2_count = (dcombo_pred == 'M2').sum()
        print(f"  ✓ {m1_count} predicciones M1, {m2_count} predicciones M2")

# Derivar Combo de DCombo
combo_pred = np.where(dcombo_pred == 'C', 'C',
                     np.where(dcombo_pred == 'MM', 'MM', 'M'))

# Diccionario de predicciones
y_pred_bottomup = {
    'DCombo': dcombo_pred,
    'Combo': combo_pred,
    'M': m_pred,
    'MM': mm_pred
}

print(f"\n✓ Predicciones Bottom-Up generadas: {len(dcombo_pred)} muestras")
print(f"\nDistribución predicha DCombo:")
print(pd.Series(dcombo_pred).value_counts().sort_index())

# Evaluar si hay etiquetas
if EVALUATE:
    results_bottomup = evaluate_predictions(y_test_dict, y_pred_bottomup, "BOTTOM-UP")


ESTRATEGIA 2: BOTTOM-UP (Ascendente/Cascada)

Descripción:
  - Nivel 1: Predice M (NO/SI)
  - Nivel 2: Si M=SI → Predice MM (NO/SI)
  - Nivel 3: Si M=SI y MM=NO → Predice M1 vs M2
  - Modelos: 3 RandomForest optimizados independientemente

Nivel 1: Prediciendo M (NO/SI)...
  ✓ 21 predicciones SI, 3 predicciones NO

Nivel 2: Prediciendo MM para 21 casos donde M=SI...
  ✓ 0 predicciones SI (MM)

Nivel 3: Prediciendo M1 vs M2 para 21 casos donde M=SI y MM=NO...
  ✓ 12 predicciones M1, 9 predicciones M2

✓ Predicciones Bottom-Up generadas: 24 muestras

Distribución predicha DCombo:
C      3
M1    12
M2     9
Name: count, dtype: int64


## 7. Estrategia 3: HYBRID (Híbrido con Validación)

In [7]:
print("\n" + "="*80)
print("ESTRATEGIA 3: HYBRID (Híbrido con Validación)")
print("="*80)
print("\nDescripción:")
print("  - Modelo principal: Top-Down (DCombo)")
print("  - Modelo de validación: Bottom-Up (M, MM)")
print("  - Regla: Si hay conflicto en M o MM Y confianza baja → usar Bottom-Up")
print(f"  - Umbral de confianza: {CONFIDENCE_THRESHOLD} (70%)")

# Obtener probabilidades del modelo Top-Down
print("\nCalculando probabilidades Top-Down...")
proba_topdown = model_topdown.predict_proba(X_test)
confidence_topdown = proba_topdown.max(axis=1)

print(f"  Confianza promedio: {confidence_topdown.mean():.4f}")
print(f"  Confianza mínima: {confidence_topdown.min():.4f}")
print(f"  Confianza máxima: {confidence_topdown.max():.4f}")
print(f"  Casos con confianza < {CONFIDENCE_THRESHOLD}: {(confidence_topdown < CONFIDENCE_THRESHOLD).sum()}")

# Inicializar con predicciones Top-Down
y_pred_hybrid = {
    'DCombo': y_pred_topdown['DCombo'].copy(),
    'Combo': y_pred_topdown['Combo'].copy(),
    'M': y_pred_topdown['M'].copy(),
    'MM': y_pred_topdown['MM'].copy()
}

# Validación cruzada: detectar conflictos y resolver con Bottom-Up
print("\n" + "-"*80)
print("VALIDACIÓN CRUZADA: Detectando y resolviendo conflictos")
print("-"*80)

conflicts_resolved = 0

for i in range(n_samples):
    # Verificar conflictos en M o MM
    conflict_m = (y_pred_topdown['M'][i] != y_pred_bottomup['M'][i])
    conflict_mm = (y_pred_topdown['MM'][i] != y_pred_bottomup['MM'][i])
    
    # Si hay conflicto Y confianza baja → usar Bottom-Up
    if (conflict_m or conflict_mm) and confidence_topdown[i] < CONFIDENCE_THRESHOLD:
        conflicts_resolved += 1
        y_pred_hybrid['M'][i] = y_pred_bottomup['M'][i]
        y_pred_hybrid['MM'][i] = y_pred_bottomup['MM'][i]
        y_pred_hybrid['DCombo'][i] = y_pred_bottomup['DCombo'][i]
        y_pred_hybrid['Combo'][i] = y_pred_bottomup['Combo'][i]

print(f"\n✓ Conflictos detectados y resueltos: {conflicts_resolved}/{n_samples}")
print(f"  Porcentaje de casos resueltos con Bottom-Up: {conflicts_resolved/n_samples*100:.2f}%")
print(f"  Porcentaje de casos con Top-Down: {(n_samples-conflicts_resolved)/n_samples*100:.2f}%")
print(f"\nDistribución predicha DCombo (Hybrid):")
print(pd.Series(y_pred_hybrid['DCombo']).value_counts().sort_index())

# Evaluar si hay etiquetas
if EVALUATE:
    results_hybrid = evaluate_predictions(y_test_dict, y_pred_hybrid, "HYBRID")


ESTRATEGIA 3: HYBRID (Híbrido con Validación)

Descripción:
  - Modelo principal: Top-Down (DCombo)
  - Modelo de validación: Bottom-Up (M, MM)
  - Regla: Si hay conflicto en M o MM Y confianza baja → usar Bottom-Up
  - Umbral de confianza: 0.7 (70%)

Calculando probabilidades Top-Down...
  Confianza promedio: 0.6452
  Confianza mínima: 0.4195
  Confianza máxima: 0.9900
  Casos con confianza < 0.7: 17

--------------------------------------------------------------------------------
VALIDACIÓN CRUZADA: Detectando y resolviendo conflictos
--------------------------------------------------------------------------------

✓ Conflictos detectados y resueltos: 7/24
  Porcentaje de casos resueltos con Bottom-Up: 29.17%
  Porcentaje de casos con Top-Down: 70.83%

Distribución predicha DCombo (Hybrid):
C      3
M1    11
M2     6
MM     4
Name: count, dtype: int64


## 8. Comparación Final de Estrategias

In [12]:
if EVALUATE:
    print("\n" + "="*80)
    print("COMPARACIÓN FINAL DE LAS 3 ESTRATEGIAS")
    print("="*80)
    
    # Crear tabla comparativa
    comparison_data = []
    
    for strategy_name, results in [
        ('Top-Down', results_topdown), 
        ('Bottom-Up', results_bottomup),
        ('Hybrid', results_hybrid)
    ]:
        for col in ['DCombo', 'Combo', 'M', 'MM']:
            comparison_data.append({
                'Estrategia': strategy_name,
                'Columna': col,
                'Accuracy': results[col]['accuracy'],
                'F1-Macro': results[col]['f1_macro']
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Mostrar tabla pivoteada
    print("\nTabla Comparativa (Accuracy):")
    print(comparison_df.pivot_table(
        index='Columna',
        columns='Estrategia',
        values='Accuracy'
    ).round(4))
    
    print("\nTabla Comparativa (F1-Macro):")
    print(comparison_df.pivot_table(
        index='Columna',
        columns='Estrategia',
        values='F1-Macro'
    ).round(4))
    
    # Promedios
    print("\n" + "="*80)
    print("RENDIMIENTO PROMEDIO POR ESTRATEGIA")
    print("="*80)
    
    avg_scores = comparison_df.groupby('Estrategia')[['Accuracy', 'F1-Macro']].mean()
    print("\n", avg_scores.round(4))
    
    # Mejor estrategia
    best_strategy = avg_scores['Accuracy'].idxmax()
    best_score = avg_scores['Accuracy'].max()
    
    print("\n" + "="*80)
    print(f"🏆 MEJOR ESTRATEGIA: {best_strategy}")
    print(f"   Accuracy promedio: {best_score:.4f}")
    print("="*80)
    
    # Guardar resultados
    comparison_df.to_csv('inference_comparison.csv', index=False)
    print("\n✓ Resultados guardados en 'inference_comparison.csv'")
else:
    print("\n⚠ No se pudo realizar comparación - no hay etiquetas (Y_test.csv)")
    print("  Las predicciones se han generado correctamente para las 3 estrategias.")


⚠ No se pudo realizar comparación - no hay etiquetas (Y_test.csv)
  Las predicciones se han generado correctamente para las 3 estrategias.


## 9. Exportar Predicciones

In [9]:
print("\n" + "="*80)
print("EXPORTANDO PREDICCIONES")
print("="*80)

# Crear DataFrame con todas las predicciones
predictions_df = pd.DataFrame({
    # Top-Down
    'TopDown_DCombo': y_pred_topdown['DCombo'],
    'TopDown_Combo': y_pred_topdown['Combo'],
    'TopDown_M': y_pred_topdown['M'],
    'TopDown_MM': y_pred_topdown['MM'],
    
    # Bottom-Up
    'BottomUp_DCombo': y_pred_bottomup['DCombo'],
    'BottomUp_Combo': y_pred_bottomup['Combo'],
    'BottomUp_M': y_pred_bottomup['M'],
    'BottomUp_MM': y_pred_bottomup['MM'],
    
    # Hybrid
    'Hybrid_DCombo': y_pred_hybrid['DCombo'],
    'Hybrid_Combo': y_pred_hybrid['Combo'],
    'Hybrid_M': y_pred_hybrid['M'],
    'Hybrid_MM': y_pred_hybrid['MM'],
    
    # Confianza Top-Down
    'TopDown_Confidence': confidence_topdown
})

# Guardar predicciones
predictions_df.to_csv('predictions_all_strategies.csv', index=False)
print("\n✓ Predicciones de las 3 estrategias guardadas en 'predictions_all_strategies.csv'")

# Mostrar resumen
print("\nRESUMEN DE PREDICCIONES:")
print(f"  Total muestras: {len(predictions_df)}")
print(f"  Columnas exportadas: {len(predictions_df.columns)}")
print(f"\nPrimeras 5 filas:")
print(predictions_df.head())

print("\n" + "="*80)
print("✓ PIPELINE DE INFERENCIA COMPLETADO")
print("="*80)


EXPORTANDO PREDICCIONES

✓ Predicciones de las 3 estrategias guardadas en 'predictions_all_strategies.csv'

RESUMEN DE PREDICCIONES:
  Total muestras: 24
  Columnas exportadas: 13

Primeras 5 filas:
  TopDown_DCombo TopDown_Combo TopDown_M TopDown_MM BottomUp_DCombo  \
0             MM            MM        SI         SI              M2   
1              C             C        NO         NO              M1   
2             MM            MM        SI         SI              M2   
3             M1             M        SI         NO              M1   
4              C             C        NO         NO              M1   

  BottomUp_Combo BottomUp_M BottomUp_MM Hybrid_DCombo Hybrid_Combo Hybrid_M  \
0              M         SI          NO            MM           MM       SI   
1              M         SI          NO            M1            M       SI   
2              M         SI          NO            MM           MM       SI   
3              M         SI          NO            M1    

## 10. Resumen y Conclusiones

In [13]:
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

print("\n✓ Modelos cargados desde 'models/'")
print(f"✓ Predicciones generadas para {n_samples} muestras")
print("\nEstrategias implementadas:")
print("  1. Top-Down: Predicción directa DCombo + mapeo jerárquico")
print("  2. Bottom-Up: Cascada de 3 niveles (M → MM → M1/M2)")
print("  3. Hybrid: Validación cruzada con resolución de conflictos")

print("\nArchivos generados:")
print("  - predictions_all_strategies.csv: Predicciones de las 3 estrategias")
if EVALUATE:
    print("  - inference_comparison.csv: Comparación de métricas")

print("\n" + "="*80)
print("PRÓXIMOS PASOS SUGERIDOS")
print("="*80)
print("""
1. Revisar 'predictions_all_strategies.csv' para análisis detallado
2. Identificar casos donde las estrategias difieren
3. Analizar confianza de Top-Down en casos conflictivos
4. Validar predicciones con expertos del dominio
5. Seleccionar estrategia final para producción
6. Implementar monitoreo continuo de rendimiento
""")

print("="*80)
print("Pipeline completado exitosamente! 🎉")
print("="*80)


RESUMEN FINAL

✓ Modelos cargados desde 'models/'
✓ Predicciones generadas para 24 muestras

Estrategias implementadas:
  1. Top-Down: Predicción directa DCombo + mapeo jerárquico
  2. Bottom-Up: Cascada de 3 niveles (M → MM → M1/M2)
  3. Hybrid: Validación cruzada con resolución de conflictos

Archivos generados:
  - predictions_all_strategies.csv: Predicciones de las 3 estrategias

PRÓXIMOS PASOS SUGERIDOS

1. Revisar 'predictions_all_strategies.csv' para análisis detallado
2. Identificar casos donde las estrategias difieren
3. Analizar confianza de Top-Down en casos conflictivos
4. Validar predicciones con expertos del dominio
5. Seleccionar estrategia final para producción
6. Implementar monitoreo continuo de rendimiento

Pipeline completado exitosamente! 🎉


In [14]:
predictions_df

,TopDown_DCombo,TopDown_Combo,TopDown_M,TopDown_MM,BottomUp_DCombo,BottomUp_Combo,BottomUp_M,BottomUp_MM,Hybrid_DCombo,Hybrid_Combo,Hybrid_M,Hybrid_MM,TopDown_Confidence
0,MM,MM,SI,SI,M2,M,SI,NO,MM,MM,SI,SI,0.980000
1,C,C,NO,NO,M1,M,SI,NO,M1,M,SI,NO,0.490405
2,MM,MM,SI,SI,M2,M,SI,NO,MM,MM,SI,SI,0.886571
3,M1,M,SI,NO,M1,M,SI,NO,M1,M,SI,NO,0.559119
4,C,C,NO,NO,M1,M,SI,NO,M1,M,SI,NO,0.513500
5,M1,M,SI,NO,M1,M,SI,NO,M1,M,SI,NO,0.487333
6,M2,M,SI,NO,M1,M,SI,NO,M2,M,SI,NO,0.419500
7,MM,MM,SI,SI,M2,M,SI,NO,MM,MM,SI,SI,0.990000
8,C,C,NO,NO,M1,M,SI,NO,M1,M,SI,NO,0.543500
9,MM,MM,SI,SI,M2,M,SI,NO,MM,MM,SI,SI,0.919071
